In [1]:
import re
from googleapiclient.discovery import build
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer


In [2]:
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Gravity\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
import os
import dotenv
dotenv.load_dotenv()

API_KEY = os.getenv("api")
MAX_COMMENTS = 300


In [8]:
def extract_video_id(url):
    pattern = r"(?:v=|\/)([0-9A-Za-z_-]{11})"
    match = re.search(pattern, url)
    return match.group(1) if match else None


In [9]:
def fetch_comments(video_id, max_comments=300):
    youtube = build("youtube", "v3", developerKey=API_KEY)
    comments = []

    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request and len(comments) < max_comments:
        response = request.execute()

        for item in response["items"]:
            text = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            comments.append(text)
            if len(comments) >= max_comments:
                break

        request = youtube.commentThreads().list_next(request, response)

    return comments


In [10]:
def analyze_sentiments(comments):
    positive = negative = neutral = 0

    for comment in comments:
        score = sia.polarity_scores(comment)["compound"]

        if score >= 0.05:
            positive += 1
        elif score <= -0.05:
            negative += 1
        else:
            neutral += 1

    return positive, negative, neutral


In [11]:
video_url = input("Enter YouTube video URL: ")
video_id = extract_video_id(video_url)

if not video_id:
    print("❌ Invalid YouTube URL")
else:
    print("🔄 Fetching comments...")
    comments = fetch_comments(video_id, MAX_COMMENTS)

    pos, neg, neu = analyze_sentiments(comments)

    print("\nRESULTS")
    print("Total comments:", len(comments))
    print("Positive:", pos)
    print("Negative:", neg)
    print("Neutral:", neu)


🔄 Fetching comments...

RESULTS
Total comments: 164
Positive: 117
Negative: 9
Neutral: 38
